# Notebook 01: Data Cleaning and Feature Engineering

This notebook focuses on **Data Preprocessing** and the construction of the core DataFrames required for our recommendation model. Our goal is to transform raw data into a structured format suitable for Machine Learning analysis.

### Objectives:
* **Binary Rating Transformation:** Developing a method to process and normalize the Rotten Tomatoes binary critic scores (Fresh/Rotten).
* **Text Vectorization:** Converting unstructured critic reviews and movie descriptions into comparable numerical representations.
* **Categorical Encoding:** Processing metadata such as **Directors** and **Genres** to enable multi-dimensional similarity comparisons.
* **Data Integrity:** Handling missing values and ensuring consistency across the dataset to prevent bias in the final model.

## Data Acquisition & Initial Exploration

In this section, we load the raw dataset into a structured format (Pandas DataFrame) to facilitate data manipulation. We will also perform a preliminary inspection to understand the dataset's schema, identify data types, and spot any immediate inconsistencies.

In [1]:
from src.book_one_utils import *
download_dataset()
critic_df, movies_df = create_dataframes()
print_preview(critic_df, movies_df)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dataset already downloaded — skipping.
=== CRITIC REVIEWS (first 5 rows) ===


,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_type,review_score,review_date,review_content
0,m/0814255,Andrew L. Urban,False,Urban Cinefile,Fresh,NaN,2010-02-06,A fantasy adventure that fuses Greek mythology...
1,m/0814255,Louise Keller,False,Urban Cinefile,Fresh,NaN,2010-02-06,"Uma Thurman as Medusa, the gorgon with a coiff..."
2,m/0814255,NaN,False,FILMINK (Australia),Fresh,NaN,2010-02-09,With a top-notch cast and dazzling special eff...
3,m/0814255,Ben McEachen,False,Sunday Mail (Australia),Fresh,3.5/5,2010-02-09,Whether audiences will get behind The Lightnin...
4,m/0814255,Ethan Alter,True,Hollywood Reporter,Rotten,NaN,2010-02-10,What's really lacking in The Lightning Thief i...




=== MOVIES METADATA (first 5 rows) ===


,rotten_tomatoes_link,movie_title,movie_info,critics_consensus,content_rating,genres,directors,authors,actors,original_release_date,...,production_company,tomatometer_status,tomatometer_rating,tomatometer_count,audience_status,audience_rating,audience_count,tomatometer_top_critics_count,tomatometer_fresh_critics_count,tomatometer_rotten_critics_count
0,m/0814255,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",Though it may seem like just another Harry Pot...,PG,"Action & Adventure, Comedy, Drama, Science Fic...",Chris Columbus,"Craig Titley, Chris Columbus, Rick Riordan","Logan Lerman, Brandon T. Jackson, Alexandra Da...",2010-02-12,...,20th Century Fox,Rotten,49.0,149.0,Spilled,53.0,254421.0,43,73,76
1,m/0878835,Please Give,Kate (Catherine Keener) and her husband Alex (...,Nicole Holofcener's newest might seem slight i...,R,Comedy,Nicole Holofcener,Nicole Holofcener,"Catherine Keener, Amanda Peet, Oliver Platt, R...",2010-04-30,...,Sony Pictures Classics,Certified-Fresh,87.0,142.0,Upright,64.0,11574.0,44,123,19
2,m/10,10,"A successful, middle-aged Hollywood songwriter...",Blake Edwards' bawdy comedy may not score a pe...,R,"Comedy, Romance",Blake Edwards,Blake Edwards,"Dudley Moore, Bo Derek, Julie Andrews, Robert ...",1979-10-05,...,Waner Bros.,Fresh,67.0,24.0,Spilled,53.0,14684.0,2,16,8
3,m/1000013-12_angry_men,12 Angry Men (Twelve Angry Men),Following the closing arguments in a murder tr...,Sidney Lumet's feature debut is a superbly wri...,NR,"Classics, Drama",Sidney Lumet,Reginald Rose,"Martin Balsam, John Fiedler, Lee J. Cobb, E.G....",1957-04-13,...,Criterion Collection,Certified-Fresh,100.0,54.0,Upright,97.0,105386.0,6,54,0
4,m/1000079-20000_leagues_under_the_sea,"20,000 Leagues Under The Sea","In 1866, Professor Pierre M. Aronnax (Paul Luk...","One of Disney's finest live-action adventures,...",G,"Action & Adventure, Drama, Kids & Family",Richard Fleischer,Earl Felton,"James Mason, Kirk Douglas, Paul Lukas, Peter L...",1954-01-01,...,Disney,Fresh,89.0,27.0,Upright,74.0,68918.0,5,24,3


# 2. Data Preprocessing: Critic Reviews Dataset

An initial analysis of the **Critic Reviews Dataset** reveals significant data sparsity and structural inconsistency. The primary challenge lies in the `review_score` feature, which presents a high variance in data formats (e.g., alphanumeric grades vs. numerical fractions) and scales.

Furthermore, the `review_type` feature provides only a binary sentiment ("Fresh" vs. "Rotten"). To maximize the predictive power of our recommendation engine, we will merge these two features into a single, high-fidelity metric.

### 2.1 Hybrid Sentiment Scoring Strategy
To establish a consistent metric for film quality, we are implementing a **Hybrid Sentiment Index ($S_{final}$)**. Instead of treating the binary label and the numerical score as independent variables, we use the score to provide the **magnitude** (intensity) of the sentiment.

#### Mathematical Normalization
1. **Raw Score Normalization ($S_{norm}$):**
   For fractional scores:
   $$S_{norm} = \frac{\text{numerator}}{\text{denominator}}$$
   For letter grades, we apply a mapping function $f(g) \in [0.1, 1.0]$.

2. **Sentiment Weighting:**
   The final score $S$ is mapped to a consistent $0-10$ scale. In cases where the numerical score is missing, we apply a **categorical fallback** based on the `review_type`:
   - **Fresh Fallback:** $S = 7.0$
   - **Rotten Fallback:** $S = 3.0$

The final integer-based metric is defined as:
$$S_{final} = \text{round}(S_{norm} \times 10)$$
Where $S_{final} \in \mathbb{Z}$ and $0 \le S_{final} \le 10$.

### 2.2 Feature Engineering & Dataset Refinement
Once the `S_final` index is calculated, the raw columns (`review_score`, `review_type`, and `review_content`) are considered redundant. To optimize memory usage and model performance, we perform a **Dimensionality Reduction** by dropping these original features and retaining only the processed index.

This results in a "Clean Dataset" where each review is represented by a single, mathematically comparable value that captures both the direction (Fresh/Rotten) and the intensity (Grade) of the critic's opinion.

### 2.3 Implementing the Score Parser
The following function, `parse_binary_score`, handles the conversion of heterogeneous formats into our normalized base.

In [2]:
dataframe_rw = init_reviews_df(movies_df,critic_df)
display(dataframe_rw.head(10))

,critic_name,rotten_tomatoes_link,final_score
3,Ben McEachen,m/0814255,7.0
6,Nick Schager,m/0814255,2.5
7,Bill Goodykoontz,m/0814255,7.0
8,Jordan Hoffman,m/0814255,8.0
9,Jim Schembri,m/0814255,6.0
10,Mark Adams,m/0814255,8.0
11,Roger Moore,m/0814255,5.0
12,David Jenkins,m/0814255,4.0
13,Joshua Tyler,m/0814255,6.0
14,Peter Paras,m/0814255,6.0


# 3. Relational & Semantic Feature Engineering: The Metadata Matrix

Having cleaned the sentiment scores, the next critical step is to transform movie metadata and narrative descriptions into a **mathematical vector space**. The objective is to define a geometric environment where we can calculate the "distance" between two films based on their shared cinematic DNA and thematic essence.

---

### 3.1 Tokenization & Atomization: Finding the "Data Atoms"
Raw metadata in the `genres` and `directors` columns are merely strings. To build a relational model, we must **atomize** these strings into unique tokens. This process strips away noise and ensures that "Action" and " Action" are recognized as the same structural component, preventing data fragmentation.

### 3.2 Statistical Significance & The Threshold Filter ($\tau$)
Including every single director would lead to an excessively sparse dataset, triggering the **Curse of Dimensionality**. We apply a **Statistical Significance Threshold**:
* **Threshold ($\tau$):** We retain only those directors who have produced a number of films $\ge \tau$.
* **Logical Reasoning:** Directors with only one film provide no "relational bridge" between nodes in your graph. Filtering them concentrates the "signal" on established patterns.

### 3.3 Semantic Deep Learning: Narrative Embeddings
Categorical tags (like "Sci-Fi") are limited. They cannot distinguish between a "space opera" and a "cyberpunk thriller." To capture the **meaning** of a film, we implement **Sentence-BERT (SBERT)** embeddings on the movie descriptions.

Unlike traditional keyword counting, **Embeddings** map sentences into a high-dimensional space where distance represents **semantic similarity**.
* *Example:* A description mentioning "time travel" and one mentioning "temporal paradox" will be placed close together, even if they share zero identical words.



### 3.4 Weighted Hybrid Matrix Integration
A naive approach assumes all features are created equal. In a recommendation engine, this is a fatal flaw. We apply **Semantic Weights ($w$)** to balance the "volume" of different data types. Genres provide the blueprint, Embeddings provide the soul, and Directors provide the execution.

#### Mathematical Definition
Let $G$ be the matrix of Genres, $D$ the matrix of Directors, and $E$ the matrix of Semantic Embeddings. The final feature matrix $M$ is defined as:

$$M = [ (G \times w_g) \quad \Vert \quad D \quad \Vert \quad (E \times w_e) ]$$

Where:
* $w_g$ is the **Genre Weight** (e.g., **2.0**), ensuring high-level category consistency.
* $w_e$ is the **Embedding Weight** (e.g., **1.5**), allowing thematic nuance to influence the result.
* $\Vert$ represents the concatenation of these diverse feature spaces into a single hybrid matrix.

---

### 3.5 Summary of the Architectural Flow
1.  **Normalization:** Clean categorical strings and narrative text.
2.  **Pruning:** Filter directors based on frequency $\tau$ to reduce dimensionality.
3.  **Encoding:** Generate dense 384-dimensional vectors for movie descriptions using SBERT.
4.  **Scaling:** Apply weights to ensure that specific relational features (Genres) don't get drowned out by dense numerical data (Embeddings).

---

### Why this works
By the end of this section, your dataset has transitioned from text to **geometry**. We have moved beyond "matching labels" to "understanding context." Two movies are now "similar" if they share an architect (Director), a structure (Genre), and a narrative heart (Embedding).

In [3]:
from src.recommender import build_feature_matrix
content_matrix = build_feature_matrix(movies_df, threshold_dir=3, weight_genres=2.0)
display(content_matrix.head(5))

Batches:   0%|          | 0/554 [00:00<?, ?it/s]

,Action & Adventure,Animation,Anime & Manga,Art House & International,Classics,Comedy,Cult Movies,Documentary,Drama,Faith & Spirituality,...,374,375,376,377,378,379,380,381,382,383
rotten_tomatoes_link,,,,,,,,,,,,,,,,,,,,,
m/0814255,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0,...,0.095316,-0.107694,-0.005921,0.069766,-0.078687,0.003852,0.156003,0.073599,0.068437,-0.015561
m/0878835,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,-0.109804,-0.043426,0.108809,0.047014,0.028263,0.093739,0.000781,0.043769,-0.010808,0.070668
m/10,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.153831,-0.154967,-0.025031,0.206110,-0.099202,0.026074,-0.088399,-0.038539,0.143202,-0.100224
m/1000013-12_angry_men,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,...,0.296300,-0.054267,0.000454,0.177189,-0.047197,-0.027004,0.346387,0.294610,0.005221,-0.050730
m/1000079-20000_leagues_under_the_sea,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.025588,-0.112626,-0.053125,-0.045511,-0.127333,0.084797,0.120427,-0.040955,-0.080480,0.059309


# 5. Data Persistence: Freezing the Engineered State

To transition from **Feature Engineering** to **Exploratory Data Analysis (EDA)**, the current architectural state must be preserved. We are shifting the data from volatile memory to the `../data/processed/` directory using a dual-format persistence strategy.

---

### 5.1 The Persistence Layer
All processed assets are stored using formats that prioritize schema integrity and computational efficiency:

* **`movies_clean.parquet` & `critic_clean.parquet`**: Standardized catalogs stored in **Parquet** format for optimized columnar access.
* **`reviews_clean.parquet`**: The unified review dataset containing the calculated $S_{final}$ metric.
* **`content_matrix.pkl`**: The high-dimensional hybrid feature space.

---

### 5.2 Technical Justification: Parquet vs. Pickle
Due to the architectural complexity of our features, a single-format approach is insufficient:

* **Parquet (DataFrames):** Selected for its ability to maintain strict data types and high compression ratios for dense tabular data.
* **Pickle (Hybrid Matrix):** **Required** to preserve the `SparseDtype` of the metadata columns (Genres/Directors). Since standard Parquet engines do not natively support Pandas' internal sparse representation, **Pickle** is used to serialize the matrix object exactly as it exists in memory, preventing forced densification and massive disk-space bloat.

---

> **Final Note:** These assets represent the "Frozen State" of our relational architecture. All subsequent analysis will derive from these processed files to ensure absolute consistency across the pipeline.

In [4]:
from src.book_one_utils import save_processed_assets

save_processed_assets(
    movies_df=movies_df,
    critic_df=critic_df,
    reviews_df=dataframe_rw,
    content_matrix=content_matrix
)

All assets saved. (content_matrix stored as .pkl)
